# PHY abstraction

> A transmission's packet error rate from its SINR, without its bits: analytic models, EESM, and 5G NR from Sionna SYS; and the draw of what arrives.

In [ ]:
#| default_exp phy_abstraction

In [ ]:
#| hide
from nbdev.showdoc import *

A *PHY abstraction* gives the packet error rate of a transmission from its SINR, without simulating its bits.
An `ErrorModel` does it for a batch of transmissions at once:

* `BPSKErrorModel` (the default) uses uncoded BPSK over AWGN. Its bit error rate is `Q(√(2 SINR))`, with
  independent bit errors, so bigger packets fail more often.
* `ThresholdErrorModel` receives a packet when the SINR is at or above a threshold.
* `EESMErrorModel` combines the subcarriers by EESM, then applies a logistic packet-error curve with your
  calibration, e.g. a Wi-Fi MCS.
* `SionnaErrorModel` (below) uses 5G NR with LDPC codes: Sionna SYS's calibrated tables, the
  subcarriers combined by EESM, and link adaptation.

The two simple models use the mean SINR over the subcarriers. The functions below are the analytic rates they use.

An `ErrorModel` is a block: called with the SINR of a step's transmissions at each subcarrier (`sinr`), their sizes
(`size_bits`), their broadcasts (`group`) and whether they are sent (`active`), it gives their packet error rates
(`per`), symbols on the air (`num_symbols`) and metadata (`info`). It rates only the transmissions sent and not
between nodes at the same place (an infinite SINR, lost never), so a new model only writes `rate`.

This is Sionna SYS's `sionna.sys.phy_abstraction` (`PHYAbstraction`), with its `sionna.sys.effective_sinr` (EESM).
Sionna's `PHYAbstraction` also draws which blocks are decoded; here `Decide` does, so a model only gives rates.

In [ ]:
#| export
from __future__ import annotations

import math
from abc import ABC, abstractmethod
from typing import List, Sequence, Tuple

import numpy as np

from comm_core.block import Block
from comm_core.link_adaptation import LinkAdaptation, NRMCSTable

In [ ]:
#| export
def bpsk_bit_error_rate(
    snr: float, # Linear signal-to-noise ratio
) -> float:
    "Bit error rate of BPSK over AWGN: Q(sqrt(2 snr))."
    return 0.5 * math.erfc(math.sqrt(max(snr, 0.0)))


def packet_error_rate(
    snr_db: float, # Signal-to-noise ratio (dB); -inf: no signal
    size_bits: int, # Packet size
    model: str = 'bpsk', # 'bpsk' (independent BPSK bit errors) or 'threshold'
    threshold_db: float = 10.0, # 'threshold': packets at or above this SNR are received
) -> float:
    "Probability that a packet is lost."
    if snr_db == math.inf:
        return 0.0
    if snr_db == -math.inf:
        return 1.0
    if model == 'threshold':
        return 0.0 if snr_db >= threshold_db else 1.0
    if model == 'bpsk':
        ber = bpsk_bit_error_rate(10 ** (snr_db / 10))
        return 1.0 - (1.0 - ber) ** max(int(size_bits), 1)
    raise ValueError(f"model: 'bpsk' or 'threshold', not {model!r}")

In [ ]:
assert packet_error_rate(30, 1000) < 1e-6 and packet_error_rate(-5, 1000) > 0.99
assert packet_error_rate(4, 10) < packet_error_rate(4, 1000)                  # bigger packets fail more often
assert packet_error_rate(9, 100, 'threshold') == 1.0 and packet_error_rate(10, 100, 'threshold') == 0.0
assert packet_error_rate(-math.inf, 8) == 1.0 and packet_error_rate(math.inf, 8) == 0.0

In [ ]:
#| export
class ErrorModel(Block, ABC):
    """A PHY abstraction: the packet error rate of transmissions, from their SINR. Subclasses write `rate`.

    The copies of a broadcast are one emission: a model that picks a rate (e.g. an MCS) gives them the same one."""

    outputs = ('per', 'num_symbols', 'info')

    def call(
        self,
        sinr: Sequence[np.ndarray], # Linear SINR of each transmission at each of its subcarriers, e.g. (n, K)
        size_bits: np.ndarray, # Packet size of each transmission
        group: Sequence | None = None, # The broadcast of each transmission, None for a unicast (None: all unicasts)
        active: np.ndarray | None = None, # Whether each one is sent (None: all)
    ) -> Tuple[np.ndarray, np.ndarray, List[dict]]:
        "(packet error rates, symbols on the air, metadata) of every transmission; `rate` gives them for the ones to rate."
        n = len(sinr)
        size = np.asarray(size_bits).astype(np.int64).reshape(n)
        per, symbols, info = np.zeros(n), size.copy(), [{} for _ in range(n)]
        rows = [k for k in range(n) if (active is None or active[k]) and not np.isinf(np.asarray(sinr[k])).any()]
        if rows:
            groups = None if group is None else [group[k] for k in rows]
            if groups is not None and all(g is None for g in groups):
                groups = None
            p, s, extra = self.rate([np.asarray(sinr[k], float).reshape(-1) for k in rows], np.maximum(size[rows], 1), groups)
            for k, pk, sk, ik in zip(rows, p, s, extra):
                per[k], symbols[k], info[k] = float(pk), int(sk), ik
        return per, symbols, info

    @abstractmethod
    def rate(
        self,
        sinr: List[np.ndarray], # Linear SINR of each transmission to rate, at each of its subcarriers (one value: flat)
        size_bits: np.ndarray, # Their packet sizes (at least 1 bit)
        group: Sequence | None = None, # Their broadcasts, None for a unicast (None: all unicasts)
    ) -> Tuple[np.ndarray, np.ndarray, List[dict]]:
        "(packet error rates, symbols on the air, extra metadata) of the transmissions."


def _mean_snr_db(sinr: np.ndarray) -> float:
    snr = float(np.mean(sinr))
    return 10 * math.log10(snr) if snr > 0 else -math.inf


class BPSKErrorModel(ErrorModel):
    "Uncoded BPSK over AWGN at the mean SINR (see `packet_error_rate`); one symbol per bit."

    def rate(self, sinr, size_bits, group=None):
        per = [packet_error_rate(_mean_snr_db(s), max(int(b), 1)) for s, b in zip(sinr, size_bits)]
        return np.array(per, float), np.asarray(size_bits, int), [{} for _ in sinr]


class ThresholdErrorModel(ErrorModel):
    "Received at or above an SINR (the mean over the subcarriers), lost below; one symbol per bit."

    def __init__(
        self,
        threshold_db: float = 10.0, # The SINR at which packets are received
    ):
        self.threshold_db = threshold_db

    def rate(self, sinr, size_bits, group=None):
        per = [packet_error_rate(_mean_snr_db(s), 1, 'threshold', self.threshold_db) for s in sinr]
        return np.array(per, float), np.asarray(size_bits, int), [{} for _ in sinr]

In [ ]:
show_doc(ErrorModel.call)

In [ ]:
show_doc(ErrorModel.rate)

In [ ]:
#| export
def eesm(
    sinr: np.ndarray, # Linear SINR at each subcarrier
    beta: float, # EESM parameter (of an MCS)
) -> float:
    "Effective SINR (linear) of SINRs at several subcarriers: -β ln(mean(exp(-SINR/β)))."
    x = -np.asarray(sinr, float) / beta
    m = x.max()
    return float(max(-beta * (m + math.log(np.mean(np.exp(x - m)))), 0.0))


class EESMErrorModel(ErrorModel):
    "EESM over the subcarriers, then a logistic packet error rate: a PHY abstraction calibrated by its parameters (e.g. a Wi-Fi MCS)."

    def __init__(
        self,
        beta: float, # EESM parameter of the MCS
        threshold_db: float, # Effective SINR at which a `reference_bytes` packet is lost 10% of the time
        slope: float = 1.5, # Steepness of the waterfall (1/dB)
        reference_bytes: int = 1000, # Packet size of `threshold_db`; other sizes scale as independent pieces
        bits_per_symbol: float = 1.0, # For the symbols on the air (the latency of `SystemLevelChannel`)
    ):
        self.beta, self.threshold_db, self.slope = beta, threshold_db, slope
        self.reference_bytes, self.bits_per_symbol = reference_bytes, bits_per_symbol

    def rate(self, sinr, size_bits, group=None):
        per, info = [], []
        for s, bits in zip(sinr, size_bits):
            eff = eesm(s, self.beta)
            eff_db = 10 * math.log10(eff) if eff > 0 else -math.inf
            if eff_db == -math.inf:
                p = 1.0
            else:
                p_ref = 1 / (1 + math.exp(min(self.slope * (eff_db - self.threshold_db) + math.log(9.0), 700)))
                p = 1 - (1 - p_ref) ** (max(int(bits), 1) / 8 / self.reference_bytes)
            per.append(p)
            info.append({'sinr_eff_db': eff_db})
        symbols = np.ceil(np.asarray(size_bits, float) / self.bits_per_symbol).astype(int)
        return np.array(per, float), symbols, info

In [ ]:
wifi = EESMErrorModel(beta=28.0, threshold_db=21.0, slope=1.5)              # ~802.11ac MCS 7 (approximate)
per, _, info = wifi([np.full(52, 10 ** 2.1), np.full(52, 10 ** 2.6)], np.array([8000, 8000]))
assert abs(per[0] - 0.1) < 1e-9 and per[1] < 0.01 and abs(info[0]['sinr_eff_db'] - 21) < 1e-9   # 10% at the threshold
rng = np.random.default_rng(0)
faded = 10 ** 2.1 * np.abs((rng.standard_normal(52) + 1j * rng.standard_normal(52)) / np.sqrt(2)) ** 2
assert eesm(faded, 28.0) < np.mean(faded)                                   # faded subcarriers count for more
assert wifi([np.zeros(52)], np.array([100]))[0][0] == 1.0
assert wifi([np.full(52, 10 ** 2.1)], np.array([16000]))[0][0] > per[0]     # bigger packets fail more

In [ ]:
per, symbols, _ = BPSKErrorModel()([np.array([10 ** 0.6]), np.array([10 ** 0.2, 10.0])], np.array([100, 100]))
assert np.allclose(per, [packet_error_rate(6, 100), packet_error_rate(10 * np.log10((10 ** 0.2 + 10) / 2), 100)])
assert symbols.tolist() == [100, 100]
# as a block: the SINRs of a step, (n, K); not rated: the ones not sent, and nodes at the same place (never lost)
per, symbols, info = BPSKErrorModel()(np.array([[1.0], [math.inf], [1.0]]), np.array([10, 10, 0]), active=np.array([True, True, False]))
assert per[0] > 0 and per[1] == per[2] == 0 and symbols.tolist() == [10, 10, 0] and info == [{}, {}, {}]
assert ThresholdErrorModel(10)([np.array([9.9]), np.array([10.0])], np.array([8, 8]))[0].tolist() == [1.0, 0.0]

## What arrives

`Decide` draws whether each packet arrives: it is lost with its packet error rate. It draws once per transmission
sent, from its own generator (`reset(seed)` makes it reproducible).

In [ ]:
#| export
class Decide(Block):
    "Whether each packet arrives: lost with its packet error rate, one draw per transmission sent."

    outputs = ('success',)

    def call(
        self,
        per: np.ndarray, # Packet error rate of each transmission
        active: np.ndarray | None = None, # Whether each one is sent (None: all)
    ) -> np.ndarray:
        success = np.zeros(len(per), dtype=bool)
        for k in range(len(per)):
            if active is None or active[k]:
                success[k] = self.rng.random() >= per[k]
        return success

In [ ]:
show_doc(Decide.call)

In [ ]:
d = Decide()
d.reset(seed=0); first = d(np.full(2000, 0.3))
assert abs(1 - first.mean() - 0.3) < 0.03                              # lost 30% of the time
d.reset(seed=0); assert (d(np.full(2000, 0.3)) == first).all()          # reproducible
assert d(np.array([0.0, 0.0]), active=np.array([True, False])).tolist() == [True, False]   # not sent: not arrived

## 5G NR, from Sionna SYS

The packet error rates of 5G NR, from [Sionna SYS](https://nvlabs.github.io/sionna/sys/index.html): LDPC-coded
transport blocks, with error rates from Sionna's calibrated tables instead of an analytic formula. It is the error
model of a `SystemLevelChannel`:

```python
channel = SystemLevelChannel(fading=..., noise_power=..., per_model=SionnaErrorModel())
```

It needs Sionna (PyTorch): `pip install 'comm-core[sionna]'`.

**How a transmission becomes a 5G NR transport block:**

1. The message is one transport block of `size_bits`.
2. An MCS (modulation and coding scheme) `m` has a modulation order `Q_m` and a code rate `R_m`. The message takes
   the fewest resource elements (one subcarrier for one OFDM symbol each) whose NR transport block holds it, at
   an effective code rate of at most 0.95, including the CRCs (`NRMCSTable`, see `link_adaptation`).
3. The SINR at each subcarrier gives one *effective* SINR, by EESM. Faded subcarriers count for more than their
   share of the mean.
4. Sionna's tables give the transport block error rate (TBLER) of that effective SINR, MCS and block size: the
   packet error rate.

**The MCS:** link adaptation (`LinkAdaptation`, see `link_adaptation`) picks, for each transmission, the highest MCS whose
TBLER is at most `bler_target`.
That's the rule of Sionna's `InnerLoopLinkAdaptation`, with each MCS given the resource elements the message needs
at it. Alternatively, `mcs_index` fixes one MCS.

**A broadcast** (`Network.broadcast`) is one emission, so its copies share one MCS: the highest whose TBLER is at
most `bler_target` at every receiver, i.e. the weakest receiver's. That's the usual rule of multicast link
adaptation. If no MCS meets the target at all of them, it's the lowest.

**The results** add `mcs`, `sinr_eff_db` and `resource_elements` to their metadata. The latency is
`resource elements / bandwidth`.

**Limits:**

* The tables are 5G NR's (its LDPC codes and MCS tables), not Wi-Fi's.
* Below the tables' range (about −5 dB), packets are lost.
* There is no HARQ soft combining. A protocol's `max_retries` resends lost messages as new transmissions.

In [ ]:
#| export
class SionnaErrorModel(ErrorModel):
    "5G NR PHY abstraction of Sionna SYS: EESM over the subcarriers, LDPC BLER tables, an MCS per transmission."

    def __init__(
        self,
        mcs_index: int | None = None, # A fixed MCS; None: link adaptation, the highest MCS meeting `bler_target`
        bler_target: float = 0.1, # Link adaptation's target TBLER
        mcs_table_index: int = 1, # 5G NR MCS table: 1 (up to 64-QAM) or 2 (up to 256-QAM)
        mcs_category: int = 0, # 0: PUSCH (uplink), 1: PDSCH (downlink)
        device: str | None = None, # Torch device; None: Sionna's default (a GPU if there is one)
        precision: str = 'single', # 'single' or 'double'
        mcs_indices: Sequence[int] | None = None, # Link adaptation's candidate MCSs; None: all of the table's
    ):
        import torch
        from sionna.sys import EESM, PHYAbstraction
        self._torch = torch
        self.phy = PHYAbstraction(precision=precision, device=device)
        self.eesm = EESM(precision=precision, device=device)
        self.device = self.phy.device
        self.dtype = torch.float64 if precision == 'double' else torch.float32
        self.link_adaptation = LinkAdaptation(bler_target)      # over the candidate MCSs, slowest to fastest
        self.bler_target, self.mcs_table_index, self.mcs_category = bler_target, mcs_table_index, mcs_category
        self.table = NRMCSTable(mcs_table_index, mcs_category, mcs_indices, device=self.device)   # the candidates
        valid = self.table.indices
        if mcs_index is not None and mcs_index not in valid:
            raise ValueError(f"mcs_index {mcs_index}: one of {valid} for MCS table {mcs_table_index}")
        self.mcs_index = mcs_index
        self.mcs = torch.tensor(valid if mcs_index is None else [mcs_index], dtype=torch.int32, device=self.device)
        self.modulation_order, self.coderate = map(list, zip(*(self.table.mcs(m) for m in self.mcs.tolist())))

    def allocation(
        self,
        num_bits: int, # Payload size
        mcs: int, # MCS index
    ) -> tuple:
        "(resource elements, TB size) of a payload at an MCS (see `NRMCSTable`)."
        return self.table.allocation(num_bits, mcs)

    @property
    def bler_target(self) -> float:
        "Link adaptation's target TBLER."
        return self.link_adaptation.target

    @bler_target.setter
    def bler_target(self, value: float) -> None:
        self.link_adaptation.target = value

    def rate(self, sinr, size_bits, group=None):
        torch = self._torch
        n, width, M = len(sinr), max(len(s) for s in sinr), len(self.mcs)
        grid = np.zeros((n, width))                        # 0: a subcarrier the transmission does not use
        for k, s in enumerate(sinr):
            grid[k, :len(s)] = np.minimum(s, 1e12)
        s = torch.as_tensor(grid, dtype=self.dtype, device=self.device)[:, None, :, None, None].expand(n, 1, width, M, 1)
        mcs = self.mcs[None].expand(n, M)
        sinr_eff = self.eesm(s, mcs, mcs_table_index=self.mcs_table_index, mcs_category=self.mcs_category)
        mcs_list = self.mcs.cpu().tolist()
        re = torch.as_tensor([[self.allocation(max(int(b), 1), m)[0] for m in mcs_list] for b in size_bits],
                             dtype=torch.int32, device=self.device)
        tbler = self.phy(mcs, sinr_eff=sinr_eff, num_allocated_re=re, mcs_table_index=self.mcs_table_index,
                         mcs_category=self.mcs_category, check_mcs_index_validity=False)[3]
        tbler = torch.nan_to_num(tbler, nan=1.0, posinf=1.0).clamp(0, 1)   # no table entry: lost
        if self.mcs_index is None:                          # the highest MCS meeting the target, else the lowest
            best = self.link_adaptation(tbler, group)       # one per broadcast
        else:
            best = torch.zeros(n, dtype=torch.long, device=self.device)
        rows = torch.arange(n, device=self.device)
        per = tbler[rows, best].cpu().numpy().astype(float)
        res = re[rows, best].cpu().numpy().astype(int)
        eff = sinr_eff[rows, best].cpu().numpy().astype(float)
        mcs_used = self.mcs[best].cpu().numpy()
        info = [{'mcs': int(m), 'sinr_eff_db': 10 * math.log10(e) if e > 0 else -math.inf, 'resource_elements': int(r)}
                for m, e, r in zip(mcs_used, eff, res)]
        return per, res, info

In [ ]:
show_doc(SionnaErrorModel.rate)

### Tests of `SionnaErrorModel`

These need Sionna, so they are flagged (`#| sionna`): `nbdev_test --flags sionna` runs them.

The packet error rate of a 1000-bit message over a flat channel, link adaptation choosing the MCS, compared
with uncoded BPSK (`BPSKErrorModel`):

In [ ]:
#| sionna
model = SionnaErrorModel()
snrs = [-6, -3, 0, 3, 6, 10, 15, 20, 25]
per, re, info = model([np.array([10 ** (x / 10)]) for x in snrs], np.full(len(snrs), 1000))
bpsk = BPSKErrorModel()([np.array([10 ** (x / 10)]) for x in snrs], np.full(len(snrs), 1000))[0]
print(' SNR   MCS  resource elements  TBLER   uncoded BPSK')
for x, p, i, b in zip(snrs, per, info, bpsk):
    print(f'{x:4d}  {i["mcs"]:4d}  {i["resource_elements"]:17d}  {p:5.3f}   {b:5.3f}')
assert per[0] == 1.0 and (per[2:] <= 0.1).all()                     # below the tables: lost; above: the target
mcs = [i['mcs'] for i in info]
assert mcs == sorted(mcs) and mcs[-1] > mcs[2] and re[-1] < re[2]      # better channels: faster MCS, fewer REs

 SNR   MCS  resource elements  TBLER   uncoded BPSK
  -6     0               4267  1.000   1.000
  -3     0               4267  1.000   1.000
   0     5               1351  0.000   1.000
   3     9                755  0.039   1.000
   6    13                523  0.097   0.908
  10    17                390  0.005   0.004
  15    22                257  0.012   0.000
  20    27                188  0.020   0.000
  25    27                188  0.020   0.000


A frequency-selective channel: EESM weighs the faded subcarriers, so the effective SINR is below the mean.

In [ ]:
#| sionna
rng = np.random.default_rng(0)
h = (rng.standard_normal(52) + 1j * rng.standard_normal(52)) / np.sqrt(2)     # Rayleigh across 52 subcarriers
fixed = SionnaErrorModel(mcs_index=15)
_, _, (flat,) = fixed([np.array([10 ** 1.5])], np.array([1000]))
_, _, (sel,) = fixed([10 ** 1.5 * np.abs(h) ** 2], np.array([1000]))
print(f"mean SINR 15 dB: effective {flat['sinr_eff_db']:.1f} dB flat, {sel['sinr_eff_db']:.1f} dB frequency-selective")
assert abs(flat['sinr_eff_db'] - 15) < 0.1 and sel['sinr_eff_db'] < flat['sinr_eff_db'] - 1
try:
    SionnaErrorModel(mcs_index=40); raise AssertionError('no error')
except ValueError:
    pass

mean SINR 15 dB: effective 15.0 dB flat, 10.4 dB frequency-selective


A broadcast to three receivers, at 25, 12 and 3 dB: alone, each would get its own MCS; together, the weakest one's.

In [ ]:
#| sionna
sinr = [np.array([10 ** (x / 10)]) for x in (25, 12, 3, 25)]
size = np.full(4, 1000)
alone = [i['mcs'] for i in model(sinr, size)[2]]
per, _, info = model(sinr, size, group=['x', 'x', 'x', None])           # the last one is a unicast
together = [i['mcs'] for i in info]
assert len(set(alone[:3])) == 3 and together[:3] == [alone[2]] * 3 and together[3] == alone[3]
assert (per[:3] <= 0.1).all()                                             # the target, at every receiver
# no MCS meets the target at a receiver below the tables: the lowest
_, _, info = model(sinr[:1] + [np.array([10 ** -1.0])], size[:2], group=['y', 'y'])
assert info[0]['mcs'] == info[1]['mcs'] == int(model.mcs[0])

In a network, the steps' transmissions go to Sionna in one call. Here, four nodes on a line take turns
(`TDMAProtocol`, each sending its newest round of messages), over a channel ray-traced at anchors with 52 subcarriers. The error model sees the per-subcarrier
SINRs:

In [ ]:
from comm_core import (Network, Node, Message, FullMeshTopology, TDMAProtocol, AnalyticalBackend,
                       SystemLevelChannel, RayTracedPathsFading, grid_points)
from comm_core.phy_abstraction import SionnaErrorModel   # the package's: the channel checks for its `ErrorModel`

In [ ]:
#| sionna
C, F = 299_792_458.0, 5.18e9
points = grid_points((0, 0), (30, 0), 0.5, height=1.0)                 # anchors every 50 cm on a 30 m corridor
v = points[None] - points[:, None]                                       # line of sight (tx, rx)
L = np.linalg.norm(v, axis=-1)
same = L == 0
L = np.where(same, 1.0, L)
paths = dict(a=np.where(same, 0, C / F / (4 * np.pi * L) * np.exp(-2j * np.pi * F * L / C))[..., None],
             tau=(L / C)[..., None], u_tx=(v / L[..., None])[..., None, :], u_rx=(-v / L[..., None])[..., None, :])
fading = RayTracedPathsFading(points, points, **paths, frequency=F, subcarriers=(np.arange(52) - 25.5) * 312.5e3)
nodes = ['a', 'b', 'c', 'd']
fading.set_positions({'a': (0.3, 0, 1), 'b': (6.1, 0, 1), 'c': (14.8, 0, 1), 'd': (29.6, 0, 1)})
channel = SystemLevelChannel(fading=fading, noise_power=10 ** ((-94 - 30) / 10), per_model=SionnaErrorModel(), seed=0,
                             interference=True)
net = Network([Node(n) for n in nodes], FullMeshTopology(nodes), TDMAProtocol(nodes, queue_size=3), AnalyticalBackend(channel))
for link in (net.topology.get_link(s, r) for s in nodes for r in nodes if s != r):
    link.tx_power_w, link.bandwidth_hz = 1e-4, 20e6                     # 0.1 mW, 20 MHz
for step in range(4):
    for s in nodes:
        for r in nodes:
            if s != r:
                net.send(Message(s, r, None, size_bits=2000))
    for res in net.step():
        t = res.transmission
        print(f'{t.sender} -> {t.receiver}: SINR {res.snr_db:5.1f} dB, MCS {res.metadata["mcs"]:2d}, '
              f'{res.metadata["resource_elements"]:5d} REs, {1e6 * res.latency:5.1f} µs, delivered {res.success}')
m = net.metrics.summary()
assert m.transmissions == 12 and m.successful >= 10

a -> b: SINR  22.0 dB, MCS 27,   376 REs,  18.8 µs, delivered True
a -> c: SINR  14.0 dB, MCS 21,   555 REs,  27.8 µs, delivered True
a -> d: SINR   7.9 dB, MCS 15,   832 REs,  41.6 µs, delivered True
b -> a: SINR  22.0 dB, MCS 27,   376 REs,  18.8 µs, delivered True
b -> c: SINR  18.5 dB, MCS 26,   391 REs,  19.6 µs, delivered True
b -> d: SINR   9.8 dB, MCS 17,   780 REs,  39.0 µs, delivered True
c -> a: SINR  14.0 dB, MCS 21,   555 REs,  27.8 µs, delivered True
c -> b: SINR  18.5 dB, MCS 26,   391 REs,  19.6 µs, delivered True
c -> d: SINR  13.9 dB, MCS 21,   555 REs,  27.8 µs, delivered True
d -> a: SINR   7.9 dB, MCS 15,   832 REs,  41.6 µs, delivered True
d -> b: SINR   9.8 dB, MCS 17,   780 REs,  39.0 µs, delivered True
d -> c: SINR  13.9 dB, MCS 21,   555 REs,  27.8 µs, delivered True


A broadcast in that network: `a` to the three others, at one MCS (unicasts to them would each get their own).

In [ ]:
#| sionna
net.reset()
for r in ('b', 'c', 'd'):
    net.send(Message('a', r, None, size_bits=2000))
unicast = {res.transmission.receiver: res.metadata['mcs'] for res in net.step()}
net.reset()
net.broadcast('a', None, size_bits=2000)
broadcast = {res.transmission.receiver: res.metadata['mcs'] for res in net.step()}
print('unicast MCSs:', unicast, ' broadcast:', broadcast)
assert len(set(unicast.values())) > 1 and set(broadcast.values()) == {min(unicast.values())}

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()